# Automatic News Headline Generator (abstractive summarization, XSum)

**Plan (about 20 minutes total):**
1. Load a seq2seq model already fine-tuned on XSum (transfer learning) and generate headlines
2. Evaluate it with ROUGE on a sample of the XSum test set
3. *Optional:* fine-tune `t5-small` on a small XSum subset ourselves and compare
4. Serve it with Flask (`app.py`)

## 1. Setup

In [ ]:
import time
import numpy as np
import torch
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

xsum = load_dataset("EdinburghNLP/xsum")
test_sample = xsum["test"].shuffle(seed=42).select(range(50))   # N=50 keeps evaluation quick
print(test_sample[0]["document"][:300], "...")

## 2. Pre-trained headline model (already fine-tuned on XSum)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

import os
from huggingface_hub import model_info

print("offline flags:", os.environ.get("HF_HUB_OFFLINE"), os.environ.get("TRANSFORMERS_OFFLINE"))

# Tries each model in order and prints the real reason if one fails
CANDIDATES = [
    "sshleifer/distilbart-xsum-12-3",
    "facebook/bart-large-xsum",
    "google/pegasus-xsum",
]
tok = model = None
for name in CANDIDATES:
    try:
        print("trying", name)
        files = [x.rfilename for x in model_info(name).siblings]
        print("  hub reachable; files:", files[:8])
        tok = AutoTokenizer.from_pretrained(name)
        model = AutoModelForSeq2SeqLM.from_pretrained(name).to(device).eval()
        MODEL_NAME = name
        print("  loaded OK")
        break
    except Exception as e:
        print("  FAILED:", type(e).__name__, str(e)[:300])

assert model is not None, "No model could be loaded; see the errors above"
print("Using:", MODEL_NAME)

def generate_headlines(model, tok, articles, prefix="", num_beams=4, batch_size=8, max_src=512):
    outs = []
    for i in range(0, len(articles), batch_size):
        batch = [prefix + a for a in articles[i:i + batch_size]]
        enc = tok(batch, return_tensors="pt", max_length=max_src,
                  truncation=True, padding=True).to(device)
        with torch.inference_mode():
            ids = model.generate(**enc, num_beams=num_beams, max_new_tokens=48,
                                 no_repeat_ngram_size=3, early_stopping=True)
        outs += [o.strip() for o in tok.batch_decode(ids, skip_special_tokens=True)]
    return outs

### Try it on your own article

In [ ]:
article = '''The city council voted on Tuesday to approve a new transport plan that will add 40 electric
buses to the network by 2028, funded partly by a government grant. Officials said the move is expected
to cut emissions and reduce journey times on the busiest routes, while residents will see new
dedicated bus lanes on three major roads.'''

print(generate_headlines(model, tok, [article])[0])

### Side-by-side with the reference summaries from XSum

In [ ]:
t0 = time.time()
preds = generate_headlines(model, tok, test_sample["document"][:5])
print(f"{time.time() - t0:.1f}s for 5 articles\n")
for p, ref in zip(preds, test_sample["summary"][:5]):
    print("PRED:", p)
    print("REF :", ref)
    print()

## 3. Evaluation with ROUGE

In [ ]:
import evaluate
rouge = evaluate.load("rouge")

def rouge_scores(preds, refs):
    r = rouge.compute(predictions=preds, references=refs, use_stemmer=True)
    return {k: round(v * 100, 2) for k, v in r.items()}

t0 = time.time()
preds_pre = generate_headlines(model, tok, test_sample["document"], num_beams=2)
print(f"{time.time() - t0:.0f}s")
scores_pre = rouge_scores(preds_pre, test_sample["summary"])
print("Pre-trained distilBART-XSum:", scores_pre)

## 4. (Optional) Fine-tune `t5-small` ourselves on a small XSum subset

Run this only if you have time left. On CPU, keep `N_TRAIN` small. This is a mini fine-tune, not a full
training run (full XSum training takes many hours), and it is meant to show the training pipeline and a
before/after comparison.

In [ ]:
from torch.utils.data import DataLoader
from transformers import DataCollatorForSeq2Seq

N_TRAIN = 500 if device == "cpu" else 3000
EPOCHS = 1
BATCH = 8
LR = 3e-4
MAX_SRC, MAX_TGT = 256, 48     # shorter inputs to keep it fast
PREFIX = "summarize: "

t5_tok = AutoTokenizer.from_pretrained("t5-small")
t5 = AutoModelForSeq2SeqLM.from_pretrained("t5-small").to(device)

train_raw = xsum["train"].shuffle(seed=1).select(range(N_TRAIN))

def preprocess(batch):
    enc = t5_tok([PREFIX + d for d in batch["document"]], max_length=MAX_SRC, truncation=True)
    enc["labels"] = t5_tok(text_target=batch["summary"], max_length=MAX_TGT, truncation=True)["input_ids"]
    return enc

train_ds = train_raw.map(preprocess, batched=True, remove_columns=train_raw.column_names)
collator = DataCollatorForSeq2Seq(t5_tok, model=t5, label_pad_token_id=-100)
loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, collate_fn=collator)

# Baseline: t5-small BEFORE fine-tuning
t5.eval()
base_preds = generate_headlines(t5, t5_tok, test_sample["document"], prefix=PREFIX,
                                num_beams=1, max_src=MAX_SRC)
print("t5-small before fine-tuning:", rouge_scores(base_preds, test_sample["summary"]))

In [ ]:
optim = torch.optim.AdamW(t5.parameters(), lr=LR, weight_decay=0.01)
total = len(loader) * EPOCHS
t5.train()
t0 = time.time()
step = 0
for epoch in range(EPOCHS):
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        loss = t5(**batch).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(t5.parameters(), 1.0)
        optim.step()
        optim.zero_grad()
        step += 1
        if step % 10 == 0 or step == total:
            print(f"step {step}/{total}  loss {loss.item():.3f}  elapsed {time.time() - t0:.0f}s")

t5.eval()
t5.save_pretrained("./t5-headline-mini")
t5_tok.save_pretrained("./t5-headline-mini")

In [ ]:
ft_preds = generate_headlines(t5, t5_tok, test_sample["document"], prefix=PREFIX,
                              num_beams=1, max_src=MAX_SRC)
scores_ft = rouge_scores(ft_preds, test_sample["summary"])
print("t5-small after mini fine-tune:", scores_ft)

import pandas as pd
print(pd.DataFrame({
    "pre-trained distilBART-XSum": scores_pre,
    "t5-small (before)": rouge_scores(base_preds, test_sample["summary"]),
    "t5-small (mini fine-tune)": scores_ft,
}).T)

## 5. Serve with Flask
Run in a terminal, from the folder containing `app.py`:

```
python app.py
```
Open http://localhost:5000 for the demo page, or POST JSON `{"article": "..."}` to `/generate`.

To serve the mini fine-tuned T5 instead: set `MODEL_NAME=./t5-headline-mini` before running (Windows cmd: `set MODEL_NAME=./t5-headline-mini`).